<a href="https://colab.research.google.com/github/0xShaswatGit22/LoRA-LLaMA-Efficient-Fine-Tuning-of-LLMs-using-QLoRA/blob/main/llama3_1_sentiment_pyspark_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === INSTALLATIONS (Run once) ===
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets tqdm pandas

print("✅ Installation completed!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-zjfmeql3/unsloth_80802351683349cd8cd9e0b83ae48fdb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-zjfmeql3/unsloth_80802351683349cd8cd9e0b83ae48fdb
  Resolved https://github.com/unslothai/unsloth.git to commit a29b4e23fd92c804ce2c46ddc4acf73d756ec823
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.4.1-py3-none-any.whl size=29525367 sha256=94c20f497eab9029d301eea86ebc9ad9a0fe6a124c5c9a8da5ad1e146e3b7871
  Stored in directory: /tmp/pip-ephem-wheel-cache-gluaxpi6/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
✅ Installation completed!


In [ ]:
import pandas as pd
import re
from datasets import Dataset
from tqdm import tqdm
import csv

# Load your CSV
df = pd.read_csv(
    "/content/Amazon_Reviews.csv",
    engine='python',
    on_bad_lines='skip',
    encoding='utf-8',
    quoting=csv.QUOTE_MINIMAL
)

print(f"✅ Loaded {len(df):,} reviews")
print("Columns:", df.columns.tolist())

# Extract numeric rating
def extract_rating(text):
    match = re.search(r'(\d+)', str(text))
    return float(match.group(1)) if match else 3.0

df['rating_numeric'] = df['Rating'].apply(extract_rating)

# Sample for T4 (you can increase to 8000 later)
sample_size = 5000
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

def create_training_example(row):
    title = str(row['Review Title'])
    text = str(row['Review Text'])
    rating = row['rating_numeric']

    full_review = f"{title}. {text}"[:1450]

    sentiment = "Positive" if rating >= 4.0 else "Negative" if rating <= 2.0 else "Neutral"

    instruction = f"""Analyze this customer review and give structured sentiment output.

Review: {full_review}"""

    response = f"""Sentiment: {sentiment}
Rating: {rating}/5.0
Key Aspects: Quality, Service, Delivery, Value
Summary: Customer had a {sentiment.lower()} experience."""

    return {
        "text": f"<|begin_of_text|>User: {instruction}\n\nAssistant: {response}<|end_of_text|>"
    }

tqdm.pandas()
examples = df_sample.progress_apply(create_training_example, axis=1).tolist()

hf_dataset = Dataset.from_list(examples)

print(f"✅ Training dataset ready with {len(hf_dataset)} examples!")

✅ Loaded 21,214 reviews
Columns: ['Reviewer Name', 'Profile Link', 'Country', 'Review Count', 'Review Date', 'Rating', 'Review Title', 'Review Text', 'Date of Experience']


100%|██████████| 5000/5000 [00:00<00:00, 35329.14it/s]


✅ Training dataset ready with 5000 examples!


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto",
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print("✅ Model + LoRA loaded successfully!")
print(model.print_trainable_parameters())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Model + LoRA loaded successfully!
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
None


In [ ]:
# === CLEAN + PROPER INSTALLATION (Run this first) ===
!pip uninstall -y unsloth unsloth_zoo

!pip install --upgrade unsloth_zoo
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install trl peft accelerate bitsandbytes datasets tqdm pandas

print("✅ Unsloth + Dependencies Installed Successfully!")

d2zH5f5cEjSF
Found existing installation: unsloth_zoo 2026.4.2
Uninstalling unsloth_zoo-2026.4.2:
  Successfully uninstalled unsloth_zoo-2026.4.2
  Using cached unsloth_zoo-2026.4.2-py3-none-any.whl.metadata (32 kB)
Using cached unsloth_zoo-2026.4.2-py3-none-any.whl (415 kB)
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-wfo0d72x/unsloth_09657341555c4a3e9b25a614080ec4c7
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-wfo0d72x/unsloth_09657341555c4a3e9b25a614080ec4c7
  Resolved https://github.com/unslothai/unsloth.git to commit a29b4e23fd92c804ce2c46ddc4acf73d756ec823
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.4.1-py3-none-any.whl size=29525367 sha256=74a61d3b6c8bc97f608866bfae498862da1f3f3590d88c3436f3bd025499195c
  Stored in directory: /tmp/pip-ephem-wheel-

In [ ]:

import torch
from unsloth import FastLanguageModel
import pandas as pd
from datasets import Dataset
from tqdm import tqdm
import re
import csv

print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Available: True
GPU Name: Tesla T4


In [ ]:
# Load your CSV
df = pd.read_csv(
    "/content/Amazon_Reviews.csv",
    engine='python',
    on_bad_lines='skip',
    encoding='utf-8',
    quoting=csv.QUOTE_MINIMAL
)

print(f"✅ Loaded {len(df):,} reviews")
print("Columns:", df.columns.tolist())

# Extract rating
def extract_rating(text):
    match = re.search(r'(\d+)', str(text))
    return float(match.group(1)) if match else 3.0

df['rating_numeric'] = df['Rating'].apply(extract_rating)

# Sample
sample_size = 5000
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

def create_example(row):
    full_review = f"{str(row['Review Title'])}. {str(row['Review Text'])}"[:1450]
    rating = row['rating_numeric']
    sentiment = "Positive" if rating >= 4.0 else "Negative" if rating <= 2.0 else "Neutral"

    instruction = f"""Analyze this customer review and provide structured sentiment analysis.

Review: {full_review}"""

    response = f"""Sentiment: {sentiment}
Rating: {rating}/5.0
Key Aspects: Quality, Service, Delivery, Value
Summary: The customer had a {sentiment.lower()} experience."""

    return {
        "text": f"<|begin_of_text|>User: {instruction}\n\nAssistant: {response}<|end_of_text|>"
    }

tqdm.pandas(desc="Creating examples")
examples = df_sample.progress_apply(create_example, axis=1).tolist()

hf_dataset = Dataset.from_list(examples)
print(f"✅ Dataset ready with {len(hf_dataset)} examples!")

✅ Loaded 21,214 reviews
Columns: ['Reviewer Name', 'Profile Link', 'Country', 'Review Count', 'Review Date', 'Rating', 'Review Title', 'Review Text', 'Date of Experience']


Creating examples: 100%|██████████| 5000/5000 [00:00<00:00, 77512.68it/s]

✅ Dataset ready with 5000 examples!


In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto",
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print("✅ Model + LoRA Loaded!")
print(model.print_trainable_parameters())

==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Model + LoRA Loaded!
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
None


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = hf_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        output_dir = "outputs",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        max_steps = 80,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "none",
    ),
)

print("🚀 Training Started...")
trainer.train()
print("🎉 Training Finished!")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

🚀 Training Started...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 80
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.33 GiB is allocated by PyTorch, and 74.89 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = hf_dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,           # Reduced from 2048 → saves big memory
    args = TrainingArguments(
        output_dir = "outputs",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,     # Reduced
        max_steps = 60,                      # Reduced for faster testing
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "none",

        # Extra memory saving options
        gradient_checkpointing = True,
        dataloader_num_workers = 0,
    ),
)

print("🚀 Starting lighter training (Lower memory usage)...")
trainer.train()
print("🎉 Training Finished!")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

🚀 Starting lighter training (Lower memory usage)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,1.676588
20,1.762755
30,1.656045
40,1.657754
50,1.705999
60,1.544105


Unsloth: Will smartly offload gradients to save VRAM!
🎉 Training Finished!


In [ ]:
# Save the trained LoRA adapters
model.save_pretrained("lora_sentiment_model")
tokenizer.save_pretrained("lora_sentiment_model")

print("✅ LoRA adapters saved successfully!")

✅ LoRA adapters saved successfully!


In [ ]:
from unsloth import FastLanguageModel

# Merge LoRA weights
model = FastLanguageModel.for_inference(model)   # Switch to inference mode

print("✅ Model ready for inference!")

✅ Model ready for inference!


In [ ]:
def analyze_review(review_text):
    prompt = f"""Analyze this customer review and provide structured sentiment analysis.

Review: {review_text}"""

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        use_cache=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Assistant:")[-1].strip()   # Extract only assistant part

# Test with some examples
test_reviews = [
    "The product is amazing! It works perfectly and delivery was super fast.",
    "Worst purchase ever. The item broke in 2 days and customer service is terrible.",
    "Average product. It's okay for the price but nothing special."
]

for review in test_reviews:
    print("="*80)
    print("Review:", review)
    print("Model Output:")
    print(analyze_review(review))
    print("="*80)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Review: The product is amazing! It works perfectly and delivery was super fast.
Model Output:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

Sentiment: Positive
Rating: 5.0/5.0
Key Aspects: Quality, Service, Delivery, Value
Summary: The customer had a positive experience.
Review: Worst purchase ever. The item broke in 2 days and customer service is terrible.
Model Output:


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentiment: Negative
Rating: 1.0/5.0
Key Aspects: Quality, Service, Delivery, Value
Summary: The customer had a negative experience.
Review: Average product. It's okay for the price but nothing special.
Model Output:
Sentiment: Neutral
Rating: 3.0/5.0
Key Aspects: Quality, Service, Delivery, Value
Summary: The customer had a neutral experience.


In [ ]:
# === SETUP PYSPARK ===
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark==3.5.1

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

spark = SparkSession.builder \
    .appName("LlamaSentimentAnalysis") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("✅ PySpark Session Created!")

✅ PySpark Session Created!


In [ ]:
def analyze_review(review_text):
    try:
        prompt = f"""Analyze this customer review and provide structured sentiment analysis.

Review: {review_text}"""

        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            use_cache=True
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract only the Assistant's response
        return response.split("Assistant:")[-1].strip()

    except Exception as e:
        return f"Error: {str(e)}"

print("✅ analyze_review function defined successfully!")

✅ analyze_review function defined successfully!


In [ ]:
# Register your existing analyze_review function as Spark UDF
sentiment_udf = udf(analyze_review, StringType())

print("✅ Spark UDF created using your fine-tuned Llama model!")

✅ Spark UDF created using your fine-tuned Llama model!


In [ ]:
# Load your Amazon Reviews CSV using PySpark
df_spark = spark.read.option("header", True) \
                     .option("multiLine", True) \
                     .option("quote", "\"") \
                     .option("escape", "\"") \
                     .csv("/content/Amazon_Reviews.csv")

# Basic cleaning
df_spark = df_spark.withColumn("clean_review",
                               col("Review Text").cast("string")) \
                   .na.drop(subset=["clean_review"])

print(f"Total Reviews Loaded: {df_spark.count():,}")

Total Reviews Loaded: 21,055


In [ ]:
# Apply your fine-tuned model on the entire dataset
result_df = df_spark.withColumn("Llama_Sentiment", sentiment_udf(col("clean_review")))

# Show sample results
print("=== Sample Predictions ===")
result_df.select(
    col("Review Title"),
    col("Review Text").substr(1, 150).alias("Review_Sample"),
    "Llama_Sentiment"
).show(8, truncate=False)

=== Sample Predictions ===
+-----------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------+
|Review Title                                   |Review_Sample                                                                                                                                         |Llama_Sentiment                       |
+-----------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------+
|A Store That Doesn't Want to Sell Anything     |I registered on the website, tried to order a laptop, entered all the details, but instead of charging me and sending the product, they froze my accou|Error: name 'tokenizer' is not defined|
|Had multiple

In [ ]:
# Sentiment Distribution
insights = result_df.groupBy("Llama_Sentiment").count().orderBy("count", ascending=False)
print("=== Sentiment Distribution ===")
insights.show(truncate=False)

# Top Positive & Negative Reviews (Optional)
result_df.filter(col("Llama_Sentiment").contains("Positive")).select("Review Title", "Review Text").show(5, truncate=100)
result_df.filter(col("Llama_Sentiment").contains("Negative")).select("Review Title", "Review Text").show(5, truncate=100)

=== Sentiment Distribution ===
+--------------------------------------+-----+
|Llama_Sentiment                       |count|
+--------------------------------------+-----+
|Error: name 'tokenizer' is not defined|21055|
+--------------------------------------+-----+

+------------+-----------+
|Review Title|Review Text|
+------------+-----------+
+------------+-----------+

+------------+-----------+
|Review Title|Review Text|
+------------+-----------+
+------------+-----------+



In [ ]:
# Save as Parquet (Industry standard for big data)
result_df.write.mode("overwrite").parquet("/content/sentiment_analysis_results")

print("✅ Results saved successfully as Parquet!")

✅ Results saved successfully as Parquet!
